# 3 - Association Rule Mining (FP-growth)

In [1]:
import numpy as np
import pandas as pd

In [2]:
# Ensure mlxtend is available for FP-growth
try:
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import fpgrowth, association_rules
except ImportError:
    import sys
    !{sys.executable} -m pip install mlxtend -q
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import fpgrowth, association_rules

In [3]:
DATA_PATH = 'data/mobile_price.csv'
TARGET_COL = 'price_range'
FEATURES = ['ram', 'int_memory', 'px_width', 'battery_power']

df = pd.read_csv(DATA_PATH)
filtered_df = df[df[TARGET_COL] == 1].copy()
selected_df = filtered_df[FEATURES].copy()

print(f'Total rows in full data: {len(df)}')
print(f'Rows with {TARGET_COL}=1: {len(filtered_df)}')
display(selected_df.head())

Total rows in full data: 2000
Rows with price_range=1: 500


,ram,int_memory,px_width,battery_power
0,2549,7,756,842
4,1411,44,1212,1821
5,1067,22,1654,1859
12,1482,33,748,1815
18,1835,49,878,1131


In [4]:
def categorize_343(series):
    """
    Split by value range using a 3:4:3 ratio:
    low = bottom 30%, medium = middle 40%, high = top 30%
    """
    min_v = series.min()
    max_v = series.max()
    value_range = max_v - min_v

    low_upper = min_v + 0.3 * value_range
    med_upper = min_v + 0.7 * value_range

    out = pd.Series(index=series.index, dtype='object')
    out[series <= low_upper] = 'low'
    out[(series > low_upper) & (series <= med_upper)] = 'medium'
    out[series > med_upper] = 'high'

    thresholds = {
        'min': min_v,
        'max': max_v,
        'range': value_range,
        'low_upper': low_upper,
        'med_upper': med_upper,
    }
    return out, thresholds

In [5]:
categorized_df = pd.DataFrame(index=selected_df.index)
threshold_rows = []

for col in FEATURES:
    cats, th = categorize_343(selected_df[col])
    categorized_df[col] = cats
    threshold_rows.append({
        'feature': col,
        'min': th['min'],
        'max': th['max'],
        'range': th['range'],
        'low_upper(30%)': th['low_upper'],
        'med_upper(70%)': th['med_upper']
    })

threshold_df = pd.DataFrame(threshold_rows)
print('Feature thresholds using 3:4:3 ratio:')
display(threshold_df)

print('Categorized samples:')
display(categorized_df.head())

Feature thresholds using 3:4:3 ratio:


,feature,min,max,range,low_upper(30%),med_upper(70%)
0,ram,387,2811,2424,1114.2,2083.8
1,int_memory,2,64,62,20.6,45.4
2,px_width,500,1998,1498,949.4,1548.6
3,battery_power,501,1996,1495,949.5,1547.5


Categorized samples:


,ram,int_memory,px_width,battery_power
0,high,low,low,low
4,medium,medium,medium,high
5,low,medium,high,high
12,medium,medium,low,high
18,medium,high,low,medium


In [6]:
# Convert each row into a transaction list like: ram_high, int_memory_low, ...
transactions = []
for _, row in categorized_df.iterrows():
    transaction = [f'{col}_{row[col]}' for col in FEATURES]
    transactions.append(transaction)

transactions_df = pd.DataFrame({'transaction': [', '.join(t) for t in transactions]})
print('First 10 transactions:')
display(transactions_df.head(10))

First 10 transactions:


,transaction
0,"ram_high, int_memory_low, px_width_low, batter..."
1,"ram_medium, int_memory_medium, px_width_medium..."
2,"ram_low, int_memory_medium, px_width_high, bat..."
3,"ram_medium, int_memory_medium, px_width_low, b..."
4,"ram_medium, int_memory_high, px_width_low, bat..."
5,"ram_high, int_memory_low, px_width_medium, bat..."
6,"ram_medium, int_memory_high, px_width_low, bat..."
7,"ram_high, int_memory_low, px_width_medium, bat..."
8,"ram_high, int_memory_medium, px_width_medium, ..."
9,"ram_medium, int_memory_high, px_width_medium, ..."


In [7]:
# One-hot encoding for FP-growth
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
trans_onehot = pd.DataFrame(te_array, columns=te.columns_)

print('Encoded transaction matrix shape:', trans_onehot.shape)
display(trans_onehot.head())

Encoded transaction matrix shape: (500, 12)


,battery_power_high,battery_power_low,battery_power_medium,int_memory_high,int_memory_low,int_memory_medium,px_width_high,px_width_low,px_width_medium,ram_high,ram_low,ram_medium
0,False,True,False,False,True,False,False,True,False,True,False,False
1,True,False,False,False,False,True,False,False,True,False,False,True
2,True,False,False,False,False,True,True,False,False,False,True,False
3,True,False,False,False,False,True,False,True,False,False,False,True
4,False,False,True,True,False,False,False,True,False,False,False,True


In [8]:
# 3(a) List all frequent patterns with support >= 0.3 using FP-growth
MIN_SUPPORT = 0.30
frequent_itemsets = fpgrowth(trans_onehot, min_support=MIN_SUPPORT, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets = frequent_itemsets.sort_values(['support', 'length'], ascending=[False, False]).reset_index(drop=True)

print(f'Frequent patterns with support >= {MIN_SUPPORT}: {len(frequent_itemsets)}')
display(frequent_itemsets)

Frequent patterns with support >= 0.3: 8


,support,itemsets,length
0,0.682,(ram_medium),1
1,0.416,(px_width_medium),1
2,0.414,(battery_power_medium),1
3,0.412,(int_memory_medium),1
4,0.318,"(ram_medium, battery_power_medium)",2
5,0.316,(int_memory_low),1
6,0.308,(battery_power_low),1
7,0.306,"(ram_medium, px_width_medium)",2


In [9]:
# 3(b) List all association rules with support >= 0.3, confidence >= 0.4, lift >= 0.8
MIN_CONFIDENCE = 0.40
MIN_RULE_SUPPORT = 0.30
MIN_LIFT = 0.80

if len(frequent_itemsets) > 0:
    rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=MIN_CONFIDENCE)
    rules = rules[
        (rules['support'] >= MIN_RULE_SUPPORT)
        & (rules['confidence'] >= MIN_CONFIDENCE)
        & (rules['lift'] >= MIN_LIFT)
    ].copy()

    if len(rules) > 0:
        rules = rules.sort_values(['lift', 'confidence', 'support'], ascending=False).reset_index(drop=True)
        rules['antecedents'] = rules['antecedents'].apply(lambda s: ', '.join(sorted(list(s))))
        rules['consequents'] = rules['consequents'].apply(lambda s: ', '.join(sorted(list(s))))
        show_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']

        print(
            f'Association rules with support >= {MIN_RULE_SUPPORT}, '
            f'confidence >= {MIN_CONFIDENCE}, lift >= {MIN_LIFT}: {len(rules)}'
        )
        display(rules[show_cols])
    else:
        print('No association rules meet the required thresholds.')
else:
    print('No frequent itemsets found, so no rules can be generated.')

Association rules with support >= 0.3, confidence >= 0.4, lift >= 0.8: 4


,antecedents,consequents,support,confidence,lift
0,battery_power_medium,ram_medium,0.318,0.768116,1.126270
1,ram_medium,battery_power_medium,0.318,0.466276,1.126270
2,px_width_medium,ram_medium,0.306,0.735577,1.078559
3,ram_medium,px_width_medium,0.306,0.448680,1.078559
